# Лаборатория к главе 17. Родословная BookLM: манифест и SHA-256 родителя

Во вторник пользователи жалуются на ответы BookLM. Чтобы откатить сервис, нужно точно знать, какие байты
отвечали: какие объединённые веса, какой граф ONNX, какой образ. В главе 17 (раздел «Родословная
модели») каждый шаг родословной записывает манифест: свой SHA-256 и SHA-256 каждого родителя. Здесь мы соберём
два звена этой цепочки, `booklm-instruct-merged-v1` → `booklm-onnx-step-v1`, и посмотрим, когда проверка
родителя срабатывает, а когда молчит.

Весов BookLM в ноутбуке нет. Вместо них два маленьких файла во временном каталоге: 131 072 байта
«объединённых весов» и несколько байт «графа». Для SHA-256 разницы нет: он смотрит только на байты.

**Вопрос опыта.**

1. Сколько бит в файле родителя нужно изменить, чтобы `verify_parents` отказалась принимать цепочку?
2. Байты родителя скопировали без изменений и записали в манифест под чужим именем. Заметит ли это
   `verify_parents`?

**Прогноз.** Впишите ответы в переменные следующей ячейки *до* того, как запустите остальные. Раздел 3
сравнит их с измерением.

Нового ноутбук не вводит: манифест с полями `id` и `parents` и проверка родителей определены в главе 17.

In [ ]:
import hashlib
import importlib.util
import json
import platform
import sys
import tempfile
from pathlib import Path

# Случайных чисел в лаборатории нет: все байты заданы явно, результат один на любой машине.
print({"python": platform.python_version(), "platform": platform.platform(),
       "hashlib": "sha256", "lab": "booklm-lineage-v1"})

# ВАШ ПРОГНОЗ — до запуска остальных ячеек
forecast_bits_to_fail = 8          # сколько бит родителя нужно испортить, чтобы проверка упала?
forecast_rename_caught = True      # заметит ли verify_parents байты под чужим именем?

## 1. Код главы из `labs/ch17`

В книге в листинге «Проверка SHA-256 родителей» оставлены только `sha256_of` и `verify_parents`. Полная
версия лежит в модуле выпуска `labs/ch17/release.py`: там же `make_manifest`, которая пишет JSON-манифест
рядом с файлом, и `verify_lineage`, которая проходит цепочку от узла вверх по всем родителям. Файл
загружается через `importlib` по явному пути, поэтому у него своё пространство имён и нет состояния от
прошлых запусков. Путь ищется от текущего каталога вверх: ноутбук работает и из `notebooks/`, и из корня
репозитория.

In [ ]:
def find_labs():
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / "labs" / "ch17"
        if (candidate / "release.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError("нет каталога labs/ch17: запускайте ноутбук из корня репозитория "
                            "или из каталога notebooks/")


LABS = find_labs()


def run_lab(name):
    # Загружает labs/ch17/<name>.py заново как отдельный модуль и возвращает его.
    spec = importlib.util.spec_from_file_location(f"labs_ch17_{name}", LABS / f"{name}.py")
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module      # свежий модуль на каждый запуск
    spec.loader.exec_module(module)
    return module


print(sorted(p.name for p in LABS.glob("*.py")))

## 2. Манифест и проверка родителей — раздел 17.1 «Родословная модели»

Файл `manifest_lineage.py` строит родословную на настоящих файлах BookLM из `models/booklm/`: копирует
токенизатор, базу, адаптер и объединённые веса во временный каталог, пишет манифест каждого звена функцией
`make_manifest`, проходит цепочку `verify_lineage` от объединённых весов вверх и портит один бит базы. У
объединённых весов два родителя, база и адаптер, и подмену базы проверка ловит, хотя адаптер цел. Все
проверки внутри — `assert`; если какая-то не сойдётся, ячейка упадёт.

In [ ]:
release = run_lab("release")          # sha256_of, make_manifest, verify_parents, verify_lineage
run_lab("manifest_lineage").main()     # печатает «manifest_lineage: OK»

Теперь соберём ту же пару звеньев сами, но оставим каталог открытым, чтобы посмотреть на манифесты. Ребёнок
хранит словарь `parents`: ключ — имя родителя, значение — SHA-256 его байтов. Рисунок «Откуда взялся выпущенный образ?» в главе 17
рисует эту запись стрелкой между узлами.

In [ ]:
RUN_ROOT = Path(tempfile.mkdtemp(prefix="booklm-17-"))
PARENT = "booklm-instruct-merged-v1"

merged = RUN_ROOT / f"{PARENT}.bin"
merged.write_bytes(bytes(range(256)) * 512)          # 131072 байт «объединённых весов»
onnx = RUN_ROOT / "booklm-onnx-step-v1.onnx"
onnx.write_bytes(b"graph-of-one-step")

parent = release.make_manifest(merged, PARENT, {}, "merge_lora")
child = release.make_manifest(onnx, "booklm-onnx-step-v1", {PARENT: merged}, "export_step")

assert merged.stat().st_size == 131_072
assert parent["parents"] == {}                        # родители этого звена вне лаборатории
assert child["parents"][PARENT] == parent["sha256"] == release.sha256_of(merged)
assert len(child["sha256"]) == 64                     # 256 бит = 64 шестнадцатеричных знака
release.verify_parents(child, {PARENT: merged})       # цепочка цела

print(Path(f"{onnx}.manifest.json").read_text())

## 3. Проверяем прогноз

Первый опыт: портим ровно один бит родителя из $131\,072 \cdot 8 = 1\,048\,576$ и смотрим, примет ли
`verify_parents` цепочку. Второй опыт: копируем байты родителя без изменений в файл с другим именем и
записываем это имя в манифест ребёнка.

In [ ]:
def parent_rejected(child_manifest, files):
    try:
        release.verify_parents(child_manifest, files)
        return False
    except ValueError:
        return True


# опыт 1: один бит
broken = RUN_ROOT / "broken-merged.bin"
data = bytearray(merged.read_bytes())
data[1000] ^= 1
broken.write_bytes(bytes(data))
measured_bits_to_fail = 1 if parent_rejected(child, {PARENT: broken}) else None
assert measured_bits_to_fail == 1

# опыт 2: те же байты под чужим именем
impostor = RUN_ROOT / "booklm-instruct-merged-v9.bin"
impostor.write_bytes(merged.read_bytes())
renamed_child = {**child, "parents": {"booklm-instruct-merged-v9": child["parents"][PARENT]}}
measured_rename_caught = parent_rejected(renamed_child, {"booklm-instruct-merged-v9": impostor})
assert measured_rename_caught is False               # байты те же, проверка молчит

print({"бит до отказа": (forecast_bits_to_fail, measured_bits_to_fail),
       "чужое имя замечено": (forecast_rename_caught, measured_rename_caught)})

Первый ответ: хватает одного бита. SHA-256 не отвечает на вопрос «насколько похожи файлы», он отвечает
только «те же ли это байты». Поэтому близость весов или логитов проверяют отдельно, метриками качества.

Второй ответ: не заметит. `verify_parents` сверяет байты, а с полем `id` в манифесте родителя ключ
словаря `parents` не сравнивает. Это и есть дыра; закрыть её предлагает задание 1 в конце ноутбука.

## 4. Намеренно сломанный вариант

Ниже версия проверки, которую легко написать по ошибке: она сравнивает только первые восемь знаков суммы,
те самые, что попадают в сообщение об ошибке. Ячейка подбирает два разных «файла» с одинаковым началом
SHA-256. На это уходит около $16^{4} = 65\,536$ попыток для четырёх знаков, поэтому мы ищем совпадение
четырёх знаков, а не восьми: принцип тот же, а ждать не нужно.

In [ ]:
def verify_parent_prefix(child_manifest, files, n=4):    # СЛОМАНО: сравнивает только начало
    for parent_id, expected in child_manifest["parents"].items():
        actual = release.sha256_of(files[parent_id])
        if actual[:n] != expected[:n]:
            raise ValueError(f"{parent_id}: начало SHA-256 не совпало")


target = child["parents"][PARENT][:4]
counter = 0
while True:                                          # перебор детерминирован: счётчик от нуля
    candidate = b"forged-" + counter.to_bytes(8, "big")
    if hashlib.sha256(candidate).hexdigest()[:4] == target:
        break
    counter += 1

forged = RUN_ROOT / "forged.bin"
forged.write_bytes(candidate)
verify_parent_prefix(child, {PARENT: forged})        # сломанная проверка пропускает подделку
assert parent_rejected(child, {PARENT: forged})      # полная проверка её ловит
print({"попыток": counter + 1, "общее начало": target})

## 5. Связь с промышленной системой

| В ноутбуке | В промышленной системе |
|---|---|
| `sha256_of(path)` | дайджест `sha256:…` слоя и манифеста образа в стандарте OCI; поле `oid sha256:…` у файлов Git LFS, в которых Hugging Face Hub хранит веса |
| `make_manifest` и файл `*.manifest.json` | карточка версии модели в реестре (например, MLflow Model Registry) с указанием прогона и исходных файлов |
| `parents`: имя родителя → его SHA-256 | ссылка на образ не по тегу (`:latest`), а по дайджесту (`image@sha256:…`) |
| `verify_parents` бросает `ValueError` | шаг конвейера сборки, который останавливает выпуск, если дайджест родителя не совпал |
| временный каталог `RUN_ROOT` | хранилище артефактов, где файл после записи не перезаписывают |

## Задания

1. Напишите проверку имени `verify_parent_names(child, files)`: для каждого ключа `child["parents"]` она
   читает манифест родителя `<файл родителя>.manifest.json` и сравнивает ключ с его полем `id`. После этого
   опыт 2 из раздела 3 должен упасть: поменяйте в нём `assert` и перезапустите ноутбук целиком.
2. Добавьте третье звено `booklm-vllm-image-v1` с родителем `booklm-instruct-merged-v1` и пройдите всю
   цепочку от образа к корню функцией `release.verify_lineage`: она проверяет каждую стрелку.
3. В разделе 4 увеличьте `n` до 5 и 6. Во сколько раз растёт число попыток? Сколько попыток понадобилось
   бы при полных 64 знаках?